## Playbook 5 — service won't start

**Symptom:** `systemctl start foo` errors, or `status` shows `failed`. There's a fixed escalation path that solves 90%+ of cases:

```
status → journalctl -u → configtest → run binary by hand → SELinux/AppArmor → daemon-reload
```

**1. Read the status** — the last log lines it prints usually *are* the answer:

```bash
sudo systemctl status foo
```

Learn the four classic messages: **`Permission denied`** (a path the service can't reach), **`Address already in use`** (another process has the port), **`No such file or directory`** (wrong path / missing binary or config), **`Failed to parse`** (config syntax error).

**2. Full logs for the unit:** `sudo journalctl -u foo -n 100 --no-pager` (module 09).

**3. Test the config** — most services ship a checker: `nginx -t`, `sshd -t`, `apachectl configtest`. The error names the file and line.

**4. Run the binary by hand** with the same command from `systemctl cat foo` — you'll see errors systemd hid, often an environment or permission issue.

**5. Check ports / ownership / MAC:** `ss -tunlp | grep :80` for the port; `ls -ld /var/lib/foo` for ownership; and on RHEL, is **SELinux** blocking it (`ausearch -m AVC -ts recent`)?

**6. Did you `daemon-reload`?** If you edited the unit file, systemd hasn't seen it until `systemctl daemon-reload` — the single most common "I changed it and it still fails" cause (module 09).
